In [ ]:
!pip install python-chess torch tqdm

In [ ]:
import os
from google.colab import drive

# 1. Подключаем твой Google Диск
drive.mount('/content/drive')

# 2. Создаем папку, откуда датасет будет читать файлы
save_dir = '/content/drive/MyDrive/ChessData/Archives'
os.makedirs(save_dir, exist_ok=True)

# 3. Список баз. Для старта берем 4 последних месяца.
# Это даст тебе около 10-15 миллионов элитных партий.
# (Ты можешь добавить сюда остальные ссылки из своего списка позже)
urls = [
    "https://database.lichess.org/standard/lichess_db_standard_rated_2026-07.pgn.zst",
    "https://database.lichess.org/standard/lichess_db_standard_rated_2026-06.pgn.zst",
    "https://database.lichess.org/standard/lichess_db_standard_rated_2026-05.pgn.zst",
    "https://database.lichess.org/standard/lichess_db_standard_rated_2026-04.pgn.zst"
]

# 4. Скачиваем каждый архив напрямую на твой Диск
for url in urls:
    filename = url.split('/')[-1]
    file_path = os.path.join(save_dir, filename)

    # Проверка, чтобы не качать заново, если сессия прервется
    if os.path.exists(file_path):
        print(f"✅ {filename} уже скачан, пропускаем.")
    else:
        print(f"⬇️ Скачиваю {filename}...")

        # Используем системный wget для максимальной скорости
        !wget -q --show-progress -O "{file_path}" "{url}"

print("🎉 Все архивы успешно сохранены на Google Диск!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⬇️ Скачиваю lichess_db_standard_rated_2026-07.pgn.zst...
/content/drive/MyDr 100%[===================>]  27.06G  20.2MB/s    in 28m 48s 
⬇️ Скачиваю lichess_db_standard_rated_2026-06.pgn.zst...
/content/drive/MyDr 100%[===================>]  26.30G  19.1MB/s    in 23m 30s 
⬇️ Скачиваю lichess_db_standard_rated_2026-05.pgn.zst...
/content/drive/MyDr 100%[===================>]  27.65G  20.5MB/s    in 24m 17s 
⬇️ Скачиваю lichess_db_standard_rated_2026-04.pgn.zst...
/content/drive/MyDr 100%[===================>]  27.31G  17.5MB/s    in 49m 47s 
🎉 Все архивы успешно сохранены на Google Диск!


In [ ]:
!pip install python-chess torch tqdm zstandard
!apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (426 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
# 1. Монтируем Google Диск
from google.colab import drive
drive.mount('/content/drive')

# 2. Ставим зависимости
!pip install python-chess zstandard -q

print("Готово. Диск смонтирован, зависимости установлены.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Готово. Диск смонтирован, зависимости установлены.


In [ ]:
# === Подготовка lichess-2400-180.pgn.zst к обучению ===
#
# Порядок действий:
#   1. Монтирует Google Drive.
#   2. Проверяет, есть ли файл уже локально или на Drive — если да, копирует
#      без повторного скачивания. Если нет — качает с Kaggle.
#   3. Проверяет целостность .zst и печатает несколько первых партий,
#      чтобы убедиться в правильном формате заголовков (WhiteElo/BlackElo/Result).
#   4. Печатает итоговый DATASET_PATH для train_chess_model.py.

import os
import io
import zipfile
import zstandard as zstd
import chess.pgn

# ================== КОНФИГУРАЦИЯ ==================
FILE_NAME = 'lichess-2400-180.pgn.zst'
KAGGLE_DATASET = 'chessmontdb/chessmont-big-dataset'   # при необходимости смените на 'dulsara/pgn-collection'

LOCAL_DIR = '/content/chess_data'
DRIVE_DIR = '/content/drive/MyDrive/ChessData/Archives'

LOCAL_PATH = os.path.join(LOCAL_DIR, FILE_NAME)
DRIVE_PATH = os.path.join(DRIVE_DIR, FILE_NAME)


def ensure_drive_mounted():
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')


def ensure_file_present():
    os.makedirs(LOCAL_DIR, exist_ok=True)
    os.makedirs(DRIVE_DIR, exist_ok=True)

    if os.path.exists(LOCAL_PATH):
        print(f"✅ Файл уже есть локально: {LOCAL_PATH}")
        return

    if os.path.exists(DRIVE_PATH):
        print(f"Копирую с Google Drive на локальный диск (быстрее для обучения)...")
        import shutil
        shutil.copy2(DRIVE_PATH, LOCAL_PATH)
        print(f"✅ Скопировано: {LOCAL_PATH}")
        return

    print("Файл не найден ни локально, ни на Drive — качаю с Kaggle...")

    # Настройка Kaggle API, если ещё не настроена
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print("Нужен kaggle.json — загрузите файл через диалог ниже.")
        from google.colab import files
        uploaded = files.upload()
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        for fname in uploaded:
            os.rename(fname, os.path.expanduser(f'~/.kaggle/{fname}'))
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    os.system('pip install kaggle -q')

    cwd = os.getcwd()
    os.chdir(LOCAL_DIR)
    ret = os.system(f'kaggle datasets download -d {KAGGLE_DATASET} -f {FILE_NAME}')
    os.chdir(cwd)

    if ret != 0:
        raise RuntimeError(
            "Скачивание не удалось. Проверьте, что kaggle.json настроен верно "
            "и что имя файла/датасета указано правильно."
        )

    # Kaggle иногда упаковывает единственный файл в .zip
    zip_path = LOCAL_PATH + '.zip'
    if os.path.exists(zip_path):
        print("Распаковываю Kaggle .zip-обёртку...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(LOCAL_DIR)
        os.remove(zip_path)

    if not os.path.exists(LOCAL_PATH):
        raise FileNotFoundError(f"После скачивания файл не найден: {LOCAL_PATH}")

    print(f"✅ Скачано: {LOCAL_PATH}")

    print("Сохраняю копию на Google Drive для сохранности...")
    import shutil
    shutil.copy2(LOCAL_PATH, DRIVE_PATH)
    print(f"✅ Копия на Drive: {DRIVE_PATH}")


def verify_archive(path, sample_games=3, min_elo=2400, min_moves=15):
    """Проверяет целостность .zst и печатает несколько партий с их эло/результатом."""
    print(f"\nПроверяю целостность архива: {path}")
    dctx = zstd.ZstdDecompressor()

    games_checked = 0
    below_threshold = 0

    with open(path, 'rb') as f:
        with dctx.stream_reader(f) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
            while games_checked < sample_games:
                game = chess.pgn.read_game(text_stream)
                if game is None:
                    break

                w_elo = game.headers.get("WhiteElo", "?")
                b_elo = game.headers.get("BlackElo", "?")
                result = game.headers.get("Result", "?")
                time_control = game.headers.get("TimeControl", "?")
                n_moves = len(list(game.mainline_moves()))

                print(f"  Партия {games_checked + 1}: WhiteElo={w_elo}, BlackElo={b_elo}, "
                      f"Result={result}, TimeControl={time_control}, ходов={n_moves}")

                try:
                    if int(w_elo) < min_elo or int(b_elo) < min_elo:
                        below_threshold += 1
                except ValueError:
                    print(f"    ⚠️  Не удалось распарсить эло как число")

                games_checked += 1

    if games_checked == 0:
        raise RuntimeError("Не удалось прочитать ни одной партии — архив повреждён или пуст.")

    if below_threshold > 0:
        print(f"\n⚠️  {below_threshold}/{games_checked} проверенных партий ниже порога {min_elo} эло. "
              f"Проверьте, тот ли файл скачан.")
    else:
        print(f"\n✅ Архив цел, заголовки в ожидаемом формате, эло проверенных партий ≥{min_elo}.")


def main():
    ensure_drive_mounted()
    ensure_file_present()
    verify_archive(LOCAL_PATH)

    size_gb = os.path.getsize(LOCAL_PATH) / (1024 ** 3)
    print(f"\n{'='*60}")
    print(f"Готово к обучению.")
    print(f"Размер файла: {size_gb:.2f} ГБ")
    print(f"Укажите в train_chess_model.py:")
    print(f"  DATASET_PATH = '{LOCAL_DIR}'")
    print(f"{'='*60}")


if __name__ == '__main__':
    main()

Копирую с Google Drive на локальный диск (быстрее для обучения)...
✅ Скопировано: /content/chess_data/lichess-2400-180.pgn.zst

Проверяю целостность архива: /content/chess_data/lichess-2400-180.pgn.zst
  Партия 1: WhiteElo=2573, BlackElo=2405, Result=1-0, TimeControl=180+0, ходов=145
  Партия 2: WhiteElo=2422, BlackElo=2403, Result=1-0, TimeControl=180+0, ходов=203
  Партия 3: WhiteElo=2405, BlackElo=2412, Result=0-1, TimeControl=180+0, ходов=86

✅ Архив цел, заголовки в ожидаемом формате, эло проверенных партий ≥2400.

Готово к обучению.
Размер файла: 12.29 ГБ
Укажите в train_chess_model.py:
  DATASET_PATH = '/content/chess_data'


In [ ]:
# Установка зависимостей (если еще не стоят в ячейке):
# !pip install python-chess zstandard torch tqdm

import os
import io
import glob
import random
import math
import torch
import torch.nn as nn
import zstandard as zstd
import chess
import chess.pgn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
from tqdm import tqdm

# ================== КОНФИГУРАЦИЯ ==================
DATASET_PATH = '/content/chess_data'
PROCESSED_DATA_DIR = '/content/chess_processed'
CHECKPOINT_DIR = '/content/drive/MyDrive/ChessData/checkpoints'

CHUNK_SIZE = 50000        # Позиций в одном бинарном файле
MIN_ELO = 2400
MIN_MOVES = 15

BATCH_SIZE = 2048         # Оптимально для 15 ГБ GPU
NUM_WORKERS = 2           # 2 воркера в Colab достаточно для чтения бинарников
NUM_RES_BLOCKS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 1000
TOTAL_STEPS = 200000
NUM_EPOCHS = 3

LOG_EVERY = 100
SAVE_EVERY = 1000
POLICY_SIZE = 4096 + 8 * 3 * 3

# ================== 1. ПРЕДПРОЦЕССИНГ (Конвертация) ==================

PIECE_TO_IDX = {
    chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
    chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5,
}
UNDERPROMOTIONS = [chess.KNIGHT, chess.BISHOP, chess.ROOK]

def board_to_tensor(board: chess.Board) -> torch.Tensor:
    flip = board.turn == chess.BLACK
    tensor = torch.zeros(12, 8, 8, dtype=torch.float32)
    for square, piece in board.piece_map().items():
        sq = chess.square_mirror(square) if flip else square
        color = (not piece.color) if flip else piece.color
        offset = 0 if color == chess.WHITE else 6
        idx = PIECE_TO_IDX[piece.piece_type] + offset
        row, col = divmod(sq, 8)
        tensor[idx, row, col] = 1.0
    return tensor

def move_to_index(move: chess.Move, flip: bool) -> int:
    frm = chess.square_mirror(move.from_square) if flip else move.from_square
    to = chess.square_mirror(move.to_square) if flip else move.to_square
    if move.promotion is not None and move.promotion != chess.QUEEN:
        from_file = chess.square_file(frm)
        to_file = chess.square_file(to)
        direction = to_file - from_file + 1
        promo_idx = UNDERPROMOTIONS.index(move.promotion)
        return 4096 + (from_file * 3 + direction) * 3 + promo_idx
    return frm * 64 + to

def save_chunk(b, p, v, idx):
    out_path = os.path.join(PROCESSED_DATA_DIR, f'chunk_{idx:05d}.pt')
    torch.save({
        'boards': torch.stack(b).half(), # half() снижает размер файла в 2 раза
        'policies': torch.tensor(p, dtype=torch.long),
        'values': torch.tensor(v, dtype=torch.float32)
    }, out_path)

def prepare_data_if_needed():
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
    existing_chunks = glob.glob(os.path.join(PROCESSED_DATA_DIR, '*.pt'))
    if len(existing_chunks) > 0:
        print(f"Данные уже сконвертированы ({len(existing_chunks)} чанков). Пропускаем предпроцессинг.")
        return

    print("Начинаем конвертацию PGN в бинарные тензоры...")
    files = sorted(glob.glob(os.path.join(DATASET_PATH, '*.zst')))
    if not files:
        raise FileNotFoundError(f"Файлы .zst не найдены в {DATASET_PATH}!")

    dctx = zstd.ZstdDecompressor()
    boards_buf, policies_buf, values_buf = [], [], []
    chunk_idx, total_positions = 0, 0

    for zst_path in files:
        print(f"Парсинг: {zst_path}")
        with open(zst_path, 'rb') as f:
            with dctx.stream_reader(f) as reader:
                text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
                while True:
                    try:
                        game = chess.pgn.read_game(text_stream)
                    except Exception:
                        continue
                    if game is None:
                        break

                    res = game.headers.get("Result", "*")
                    if res == "1-0": game_result = 1.0
                    elif res == "0-1": game_result = -1.0
                    elif res == "1/2-1/2": game_result = 0.0
                    else: continue

                    try:
                        w_elo, b_elo = int(game.headers.get("WhiteElo", 0)), int(game.headers.get("BlackElo", 0))
                    except ValueError: continue
                    if w_elo < MIN_ELO or b_elo < MIN_ELO: continue

                    moves = list(game.mainline_moves())
                    if len(moves) < MIN_MOVES: continue

                    board = game.board()
                    for move in moves:
                        flip = board.turn == chess.BLACK
                        boards_buf.append(board_to_tensor(board))
                        policies_buf.append(move_to_index(move, flip))
                        values_buf.append([game_result if board.turn == chess.WHITE else -game_result])
                        board.push(move)

                        if len(boards_buf) >= CHUNK_SIZE:
                            save_chunk(boards_buf, policies_buf, values_buf, chunk_idx)
                            total_positions += len(boards_buf)
                            chunk_idx += 1
                            boards_buf, policies_buf, values_buf = [], [], []

    if boards_buf:
        save_chunk(boards_buf, policies_buf, values_buf, chunk_idx)
        total_positions += len(boards_buf)

    print(f"Конвертация завершена! Позиций: {total_positions}, чанков: {chunk_idx + 1}")

# ================== 2. ДАТАСЕТ И АРХИТЕКТУРА ==================

class PreprocessedChessDataset(IterableDataset):
    def __init__(self, folder_path):
        self.files = sorted(glob.glob(os.path.join(folder_path, '*.pt')))
        if not self.files:
            raise FileNotFoundError(f"Файлы .pt не найдены в {folder_path}")

    def __iter__(self):
        worker_info = get_worker_info()
        worker_files = self.files if worker_info is None else self.files[worker_info.id::worker_info.num_workers]
        random.shuffle(worker_files)

        for path in worker_files:
            data = torch.load(path)
            boards = data['boards'].float() # Возвращаем во float32 для обучения
            policies = data['policies']
            values = data['values']

            indices = torch.randperm(boards.size(0))
            for i in indices:
                yield boards[i], policies[i], values[i]

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.conv(x))

class DualHeadChessNet(nn.Module):
    def __init__(self, num_res_blocks=10, policy_size=POLICY_SIZE):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv2d(12, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(128) for _ in range(num_res_blocks)])

        self.policy_head = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, policy_size),
        )

        self.value_head = nn.Sequential(
            nn.Conv2d(128, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.input_conv(x)
        x = self.res_blocks(x)
        return self.policy_head(x), self.value_head(x)

# ================== 3. ОБУЧЕНИЕ ==================

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, (TOTAL_STEPS - WARMUP_STEPS)))
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

def train():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    latest_path = os.path.join(CHECKPOINT_DIR, 'dual_chess_net_latest.pth')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Запуск на {device} с размером батча {BATCH_SIZE}")

    if device.type == 'cuda':
        torch.backends.cudnn.benchmark = True

    model = DualHeadChessNet(num_res_blocks=NUM_RES_BLOCKS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    policy_criterion = nn.CrossEntropyLoss()
    value_criterion = nn.MSELoss()

    step = 0
    start_epoch = 0
    if os.path.exists(latest_path):
        ckpt = torch.load(latest_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        scaler.load_state_dict(ckpt['scaler_state_dict'])
        step, start_epoch = ckpt.get('step', 0), ckpt.get('epoch', 0)
        print(f"Восстановление из чекпоинта: step={step}, epoch={start_epoch}")

    dataset = PreprocessedChessDataset(PROCESSED_DATA_DIR)

    for epoch in range(start_epoch, NUM_EPOCHS):
        dataloader = DataLoader(
            dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
            pin_memory=True, prefetch_factor=2
        )

        model.train()
        running_policy_loss, running_value_loss = 0.0, 0.0
        pbar = tqdm(dataloader, desc=f"Эпоха {epoch + 1}/{NUM_EPOCHS}")

        for boards, policy_targets, value_targets in pbar:
            boards = boards.to(device, non_blocking=True)
            policy_targets = policy_targets.to(device, non_blocking=True)
            value_targets = value_targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                policy_preds, value_preds = model(boards)
                policy_loss = policy_criterion(policy_preds, policy_targets)
                value_loss = value_criterion(value_preds, value_targets)
                loss = policy_loss + value_loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_policy_loss += policy_loss.item()
            running_value_loss += value_loss.item()
            step += 1

            if step % LOG_EVERY == 0:
                pbar.set_postfix({
                    'pol': f"{running_policy_loss / LOG_EVERY:.3f}",
                    'val': f"{running_value_loss / LOG_EVERY:.3f}",
                    'lr': f"{scheduler.get_last_lr()[0]:.2e}",
                })
                running_policy_loss, running_value_loss = 0.0, 0.0

            if step % SAVE_EVERY == 0:
                tmp_path = latest_path + '.tmp'
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'scaler_state_dict': scaler.state_dict(),
                    'step': step, 'epoch': epoch,
                }, tmp_path)
                os.replace(tmp_path, latest_path)

        # Сохранение в конце эпохи
        tmp_path = latest_path + '.tmp'
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'step': step, 'epoch': epoch + 1,
        }, tmp_path)
        os.replace(tmp_path, latest_path)

if __name__ == '__main__':
    prepare_data_if_needed()
    train()

Начинаем конвертацию PGN в бинарные тензоры...
Парсинг: /content/chess_data/lichess-2400-180.pgn.zst


In [ ]:
# === MCTS-движок поверх Dual-Head шахматной сети ===
#
# Установка зависимостей:
#   pip install python-chess torch
#
# Режимы запуска:
#   python mcts_chess_engine.py --checkpoint dual_chess_net_latest.pth --mode console
#   python mcts_chess_engine.py --checkpoint dual_chess_net_latest.pth --mode uci

import argparse
import math
import random
import sys
import time

import chess
import torch
import torch.nn as nn
import torch.nn.functional as F

PIECE_TO_IDX = {
    chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
    chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5,
}
UNDERPROMOTIONS = [chess.KNIGHT, chess.BISHOP, chess.ROOK]
POLICY_SIZE = 4096 + 8 * 3 * 3  # 4168


def board_to_tensor(board: chess.Board) -> torch.Tensor:
    flip = board.turn == chess.BLACK
    tensor = torch.zeros(12, 8, 8, dtype=torch.float32)
    for square, piece in board.piece_map().items():
        sq = chess.square_mirror(square) if flip else square
        color = (not piece.color) if flip else piece.color
        offset = 0 if color == chess.WHITE else 6
        idx = PIECE_TO_IDX[piece.piece_type] + offset
        row, col = divmod(sq, 8)
        tensor[idx, row, col] = 1.0
    return tensor


def move_to_index(move: chess.Move, flip: bool) -> int:
    frm = chess.square_mirror(move.from_square) if flip else move.from_square
    to = chess.square_mirror(move.to_square) if flip else move.to_square

    if move.promotion is not None and move.promotion != chess.QUEEN:
        from_file = chess.square_file(frm)
        to_file = chess.square_file(to)
        direction = to_file - from_file + 1
        promo_idx = UNDERPROMOTIONS.index(move.promotion)
        extra_idx = (from_file * 3 + direction) * 3 + promo_idx
        return 4096 + extra_idx

    return frm * 64 + to


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.conv(x))


class DualHeadChessNet(nn.Module):
    def __init__(self, num_res_blocks=10, policy_size=POLICY_SIZE):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv2d(12, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(128) for _ in range(num_res_blocks)])

        self.policy_head = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, policy_size),
        )

        self.value_head = nn.Sequential(
            nn.Conv2d(128, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.input_conv(x)
        x = self.res_blocks(x)
        return self.policy_head(x), self.value_head(x)


def load_model(checkpoint_path, device, num_res_blocks=10):
    model = DualHeadChessNet(num_res_blocks=num_res_blocks).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    return model


class MCTSNode:
    __slots__ = ('parent', 'move', 'prior', 'children', 'visit_count', 'value_sum', 'is_expanded')

    def __init__(self, parent=None, move=None, prior=0.0):
        self.parent = parent
        self.move = move
        self.prior = prior
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0.0
        self.is_expanded = False

    @property
    def q_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count


class MCTS:
    def __init__(self, model, device, c_puct=1.5, dirichlet_alpha=0.3, dirichlet_epsilon=0.25):
        self.model = model
        self.device = device
        self.c_puct = c_puct
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_epsilon = dirichlet_epsilon

    @torch.no_grad()
    def evaluate(self, board: chess.Board):
        flip = board.turn == chess.BLACK
        tensor = board_to_tensor(board).unsqueeze(0).to(self.device)
        policy_logits, value = self.model(tensor)
        policy_logits = policy_logits.squeeze(0)
        value = value.item()

        legal_moves = list(board.legal_moves)
        indices = [move_to_index(m, flip) for m in legal_moves]
        idx_tensor = torch.tensor(indices, device=self.device, dtype=torch.long)
        legal_logits = policy_logits[idx_tensor]
        probs = F.softmax(legal_logits, dim=0).cpu().tolist()

        policy = {m: p for m, p in zip(legal_moves, probs)}
        return policy, value

    def _expand(self, node: MCTSNode, policy: dict):
        for move, prob in policy.items():
            node.children[move] = MCTSNode(parent=node, move=move, prior=prob)
        node.is_expanded = True

    def _add_dirichlet_noise(self, node: MCTSNode):
        moves = list(node.children.keys())
        if not moves:
            return
        noise = [random.gammavariate(self.dirichlet_alpha, 1.0) for _ in moves]
        total = sum(noise)
        noise = [n / total for n in noise]
        for move, n in zip(moves, noise):
            child = node.children[move]
            child.prior = child.prior * (1 - self.dirichlet_epsilon) + n * self.dirichlet_epsilon

    def _select_child(self, node: MCTSNode):
        total_visits = sum(c.visit_count for c in node.children.values())
        sqrt_total = math.sqrt(total_visits) if total_visits > 0 else 1.0
        best_score, best_move, best_child = -float('inf'), None, None
        for move, child in node.children.items():
            u = self.c_puct * child.prior * sqrt_total / (1 + child.visit_count)
            score = child.q_value + u
            if score > best_score:
                best_score, best_move, best_child = score, move, child
        return best_move, best_child

    @staticmethod
    def _terminal_value(board: chess.Board) -> float:
        if board.is_checkmate():
            return -1.0
        return 0.0

    @staticmethod
    def _backup(path, value):
        for node in reversed(path):
            node.visit_count += 1
            node.value_sum += value
            value = -value

    def run(self, board: chess.Board, num_simulations=None, time_budget=None, add_noise=False):
        root = MCTSNode()
        if board.is_game_over():
            return root

        policy, _ = self.evaluate(board)
        self._expand(root, policy)
        if add_noise:
            self._add_dirichlet_noise(root)

        start = time.time()
        sims_done = 0
        target_sims = num_simulations if num_simulations is not None else (400 if time_budget is None else None)

        while True:
            if target_sims is not None and sims_done >= target_sims:
                break
            if time_budget is not None and (time.time() - start) >= time_budget:
                break

            node = root
            sim_board = board.copy()
            path = [node]

            while node.is_expanded and node.children:
                move, node = self._select_child(node)
                sim_board.push(move)
                path.append(node)

            if sim_board.is_game_over():
                leaf_value = self._terminal_value(sim_board)
            else:
                policy, leaf_value = self.evaluate(sim_board)
                self._expand(node, policy)

            self._backup(path, leaf_value)
            sims_done += 1

        return root

    @staticmethod
    def best_move(root: MCTSNode, temperature=0.0):
        if not root.children:
            return None
        if temperature <= 1e-8:
            return max(root.children.items(), key=lambda kv: kv[1].visit_count)[0]
        moves = list(root.children.keys())
        visits = [root.children[m].visit_count for m in moves]
        weights = [v ** (1.0 / temperature) for v in visits]
        total = sum(weights)
        weights = [w / total for w in weights]
        return random.choices(moves, weights=weights, k=1)[0]


def get_best_move(mcts: MCTS, board: chess.Board, num_simulations=None, time_budget=None, temperature=0.0):
    root = mcts.run(board, num_simulations=num_simulations, time_budget=time_budget)
    return mcts.best_move(root, temperature=temperature), root


def run_console(model, device, simulations):
    board = chess.Board()
    mcts = MCTS(model, device)

    human_color = None
    while human_color is None:
        answer = input("Играть за белых или чёрных? (w/b): ").strip().lower()
        if answer in ('w', 'white'):
            human_color = chess.WHITE
        elif answer in ('b', 'black'):
            human_color = chess.BLACK

    print(board)
    print()

    while not board.is_game_over():
        if board.turn == human_color:
            move_str = input("Твой ход (например e2e4, или 'e7e8q' для превращения): ").strip()
            try:
                move = chess.Move.from_uci(move_str)
                if move not in board.legal_moves:
                    print("Нелегальный ход, попробуй ещё раз.")
                    continue
            except ValueError:
                print("Не понял ход, формат: e2e4 (UCI).")
                continue
            board.push(move)
        else:
            print(f"Движок думает ({simulations} симуляций)...")
            t0 = time.time()
            move, root = get_best_move(mcts, board, num_simulations=simulations)
            elapsed = time.time() - t0
            top = sorted(root.children.items(), key=lambda kv: -kv[1].visit_count)[:3]
            top_str = ", ".join(f"{m.uci()}({c.visit_count})" for m, c in top)
            print(f"Ход движка: {move.uci()}  [{elapsed:.1f}с, топ: {top_str}]")
            board.push(move)

        print()
        print(board)
        print()

    print("Игра окончена:", board.result())


def parse_go_params(line):
    parts = line.split()
    params = {}
    i = 1
    while i < len(parts):
        key = parts[i]
        if key in ('movetime', 'wtime', 'btime', 'winc', 'binc', 'nodes', 'depth'):
            try:
                params[key] = int(parts[i + 1])
            except (IndexError, ValueError):
                pass
            i += 2
        else:
            i += 1
    return params


def compute_budget(params, board):
    if 'nodes' in params:
        return params['nodes'], None
    if 'movetime' in params:
        return None, params['movetime'] / 1000.0
    time_key = 'wtime' if board.turn == chess.WHITE else 'btime'
    if time_key in params:
        remaining = params[time_key] / 1000.0
        return None, max(0.2, remaining / 30.0)
    return None, 2.0


def parse_position(line):
    parts = line.split()
    board = chess.Board()
    if len(parts) < 2:
        return board
    if parts[1] == 'startpos':
        idx = 2
    elif parts[1] == 'fen':
        fen = ' '.join(parts[2:8])
        board = chess.Board(fen)
        idx = 8
    else:
        idx = 2
    if idx < len(parts) and parts[idx] == 'moves':
        for mv in parts[idx + 1:]:
            board.push_uci(mv)
    return board


def run_uci(model, device, default_simulations):
    board = chess.Board()
    mcts = MCTS(model, device)

    while True:
        line = sys.stdin.readline()
        if not line:
            break
        line = line.strip()
        if not line:
            continue

        if line == 'uci':
            print("id name DualHeadChessNet-MCTS")
            print("id author user")
            print("uciok")
        elif line == 'isready':
            print("readyok")
        elif line == 'ucinewgame':
            board = chess.Board()
        elif line.startswith('position'):
            board = parse_position(line)
        elif line.startswith('go'):
            params = parse_go_params(line)
            sims, time_budget = compute_budget(params, board)
            if sims is None and time_budget is None:
                sims = default_simulations
            move, _ = get_best_move(mcts, board, num_simulations=sims, time_budget=time_budget)
            if move is None:
                print("bestmove 0000")
            else:
                print(f"bestmove {move.uci()}")
        elif line == 'quit':
            break

        sys.stdout.flush()


CHECKPOINT_PATH = '/content/drive/MyDrive/ChessData/checkpoints/dual_chess_net_latest.pth'
RES_BLOCKS = 10
SIMULATIONS = 400

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Загружаю модель на {device}...")
model = load_model(CHECKPOINT_PATH, device, num_res_blocks=RES_BLOCKS)

run_console(model, device, SIMULATIONS)